# PyMol Command Generator

The purpose of this notebook is to generate PyMol from the DATA files from the MD trajectory Analysis pipeline.

We use the RMSF calculation to determine __High Mobility residues__, by identifying residues with a movmeent ≥ 1.5Å.

In [3]:
import os
import glob
import pandas as pd

In [4]:
# Define the base directory where your DATA folder is located.
# Assuming your Jupyter notebook is in the 'AT1R' project root folder.
data_dir = "AT2R/DATA"

# Find all high mobility CSV files (handles both 'High' and 'high' naming variations)
search_pattern = os.path.join(data_dir, "**", "*[Hh]igh_[Mm]obility_[Rr]eceptor.csv")
csv_files = glob.glob(search_pattern, recursive=True)

# Quick check to ensure the path is working
if not csv_files:
    print("No matching CSV files found. Please check your path.")
else:
    print(f"Found {len(csv_files)} systems to process.")

Found 4 systems to process.


In [5]:
print("### COPY THE LINES BELOW INTO THE PYMOL TERMINAL ###\n")

# --- GLOBAL VISUALIZATION SETTINGS ---


# Define a robust palette of distinct PyMOL colors
color_palette = [
    "red", "green", "blue", "yellow", "magenta", "cyan", 
    "orange", "purple", "pink", "teal", "salmon", "limon"
]

for file_path in sorted(csv_files):
    system_name = os.path.basename(os.path.dirname(file_path))
    df = pd.read_csv(file_path)
    
    if 'Residue' in df.columns:
        # Extract unique residues, sort them mathematically
        residues = sorted(df['Residue'].dropna().astype(int).unique().tolist())
        
        if not residues:
            continue
            
        # --- ALGORITHM: Group into contiguous segments ---
        segments = []
        current_segment = [residues[0]]
        
        for res in residues[1:]:
            if res == current_segment[-1] + 1:
                current_segment.append(res)
            else:
                segments.append(current_segment)
                current_segment = [res]
        segments.append(current_segment)
        # -------------------------------------------------

# --- NEW: Create a readable summary list of the segments ---
        segment_summary = []
        for segment in segments:
            if len(segment) > 1:
                segment_summary.append(f"{segment[0]}-{segment[-1]}")
            else:
                segment_summary.append(f"{segment[0]}")
        
        # OLD: summary_str = ", ".join(segment_summary)
        # NEW: Join them with a newline and a comment dash
        summary_str = "\n".join(segment_summary)

# Print the Header and the Summary
        print(f"# --- {system_name} (Found {len(segments)} discrete segments) ---")
        print(f"Segments:\n{summary_str}")
        print("# --- Global Setup ---")
        print("bg_color white")
        print("as cartoon, all")
        print("color grey90, all")
        print("set cartoon_side_chain_helper, on") 
        print("hide hydrogens")
        print("show hydrogens, (elem N,O,S) extend 1")
        print()
        
        # Generate the PyMOL commands
        for i, segment in enumerate(segments):

            if len(segment) > 1:
                res_string = f"{segment[0]}-{segment[-1]}"
            else:
                res_string = f"{segment[0]}"
                
            sel_name = f"mob_{system_name.lower()}_seg{i+1}"
            color_name = color_palette[i % len(color_palette)]
            
            print(f"select {sel_name}, resi {res_string}")
            print(f"show sticks, {sel_name}")
            print(f"color {color_name}, {sel_name}")
            
        print(f"group {system_name}_HighMobility, mob_{system_name.lower()}_seg*")
        print() 
    else:
        print(f"# Warning: 'Residue' column not found in {file_path}\n")

### COPY THE LINES BELOW INTO THE PYMOL TERMINAL ###

# --- A18 (Found 7 discrete segments) ---
Segments:
1-6
38-42
77-79
114-123
207-216
218-219
285-306
# --- Global Setup ---
bg_color white
as cartoon, all
color grey90, all
set cartoon_side_chain_helper, on
hide hydrogens
show hydrogens, (elem N,O,S) extend 1

select mob_a18_seg1, resi 1-6
show sticks, mob_a18_seg1
color red, mob_a18_seg1
select mob_a18_seg2, resi 38-42
show sticks, mob_a18_seg2
color green, mob_a18_seg2
select mob_a18_seg3, resi 77-79
show sticks, mob_a18_seg3
color blue, mob_a18_seg3
select mob_a18_seg4, resi 114-123
show sticks, mob_a18_seg4
color yellow, mob_a18_seg4
select mob_a18_seg5, resi 207-216
show sticks, mob_a18_seg5
color magenta, mob_a18_seg5
select mob_a18_seg6, resi 218-219
show sticks, mob_a18_seg6
color cyan, mob_a18_seg6
select mob_a18_seg7, resi 285-306
show sticks, mob_a18_seg7
color orange, mob_a18_seg7
group A18_HighMobility, mob_a18_seg*

# --- A19 (Found 6 discrete segments) ---
Segments:
1-